# Hydrogen Bond Analysis - MD Simulation
## 264THM-PPARG and Luteolin-PDE5A Complexes

## Step 1: Install GROMACS 2024

In [ ]:
%%bash
if [ ! -f /root/micromamba ]; then
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    mv bin/micromamba /root/micromamba
    rm -rf bin
fi
/root/micromamba create -y -n gmx -c conda-forge gromacs=2024 > /dev/null 2>&1
/root/micromamba run -n gmx gmx --version | head -5

In [ ]:
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt

GMX = '/root/micromamba run -n gmx gmx'
DATASET_PATH = '/kaggle/input/md-hbond-analysis'
WORK_DIR = '/kaggle/working'

print('Dataset:')
for c in ['264THM_PPARG', 'Luteolin_PDE5A']:
    print(f'  {c}: {"OK" if os.path.exists(f"{DATASET_PATH}/{c}") else "NOT FOUND"}')

## Step 2: Check hbond-legacy availability

In [ ]:
# Check if hbond-legacy exists (the old version that uses interactive groups)
result = subprocess.run(f'{GMX} hbond-legacy -h 2>&1 | head -20', shell=True, capture_output=True, text=True)
print('hbond-legacy check:')
print(result.stdout + result.stderr)

## Step 3: H-Bond Analysis using hbond-legacy

In [ ]:
def run_hbond_legacy(name):
    """Use gmx hbond-legacy with interactive group selection"""
    d = f'{DATASET_PATH}/{name}'
    tpr, xtc = f'{d}/md.tpr', f'{d}/trajectory_clean.xtc'
    os.makedirs(f'{WORK_DIR}/{name}', exist_ok=True)
    out = f'{WORK_DIR}/{name}/hbond_num.xvg'
    
    print(f'\n{"="*60}')
    print(f'{name} H-Bond Analysis (legacy)')
    print(f'{"="*60}')
    
    # Group 1 = Protein, Group 13 = UNL (ligand)
    cmd = f'printf "1\n13\n" | {GMX} hbond-legacy -f {xtc} -s {tpr} -num {out} 2>&1'
    print(f'Running: gmx hbond-legacy with Protein (1) and UNL (13)')
    
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = r.stdout + r.stderr
    
    # Print key info
    for line in output.split('\n'):
        if any(x in line.lower() for x in ['found', 'hbond', 'donor', 'acceptor', 'error', 'warning', 'frame']):
            print(line)
    
    if os.path.exists(out):
        print(f'\nSUCCESS: {out}')
        return out
    else:
        print('\nH-bond legacy failed')
        return None

# Try hbond-legacy first
results = {}
for name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    results[name] = run_hbond_legacy(name)

## Step 4: Alternative - New hbond with simple syntax

In [ ]:
def run_hbond_new(name):
    """Use new gmx hbond with -r and -t only"""
    d = f'{DATASET_PATH}/{name}'
    tpr, xtc = f'{d}/md.tpr', f'{d}/trajectory_clean.xtc'
    os.makedirs(f'{WORK_DIR}/{name}', exist_ok=True)
    out = f'{WORK_DIR}/{name}/hbond_num.xvg'
    
    if os.path.exists(out):
        print(f'{name}: Already have H-bond data')
        return out
    
    print(f'\n{"="*60}')
    print(f'{name} H-Bond Analysis (new syntax)')
    print(f'{"="*60}')
    
    # Simple syntax with just -r and -t
    cmd = f'{GMX} hbond -f {xtc} -s {tpr} -num {out} -r "protein" -t "resname UNL" 2>&1'
    print(f'Running: gmx hbond -r "protein" -t "resname UNL"')
    
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = r.stdout + r.stderr
    
    for line in output.split('\n')[-30:]:
        if line.strip():
            print(line)
    
    if os.path.exists(out):
        print(f'\nSUCCESS: {out}')
        return out
    return None

# Try new hbond if legacy failed
for name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    if results.get(name) is None:
        results[name] = run_hbond_new(name)

## Step 5: Fallback - Contact Analysis

In [ ]:
def run_contact(name):
    d = f'{DATASET_PATH}/{name}'
    tpr, xtc = f'{d}/md.tpr', f'{d}/trajectory_clean.xtc'
    os.makedirs(f'{WORK_DIR}/{name}', exist_ok=True)
    out = f'{WORK_DIR}/{name}/numcont.xvg'
    
    cmd = f'printf "1\n13\n" | {GMX} mindist -f {xtc} -s {tpr} -od {WORK_DIR}/{name}/mindist.xvg -on {out} -d 0.35 2>&1'
    print(f'\n=== {name} Contact Analysis ===')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if os.path.exists(out):
        print(f'SUCCESS: {out}')
        return out
    return None

contact_results = {}
for name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    if results.get(name) is None:
        contact_results[name] = run_contact(name)

## Step 6: Visualization

In [ ]:
def parse_xvg(f):
    t, v = [], []
    if not os.path.exists(f): return np.array([]), np.array([])
    for line in open(f):
        if not line.startswith(('#','@')):
            p = line.split()
            if len(p) >= 2:
                t.append(float(p[0]))
                v.append(float(p[1]))
    return np.array(t), np.array(v)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
summary = {}

for idx, name in enumerate(['264THM_PPARG', 'Luteolin_PDE5A']):
    hb_file = f'{WORK_DIR}/{name}/hbond_num.xvg'
    cont_file = f'{WORK_DIR}/{name}/numcont.xvg'
    
    if os.path.exists(hb_file):
        t, v = parse_xvg(hb_file)
        ylabel = 'H-bonds'
        data_type = 'H-bonds'
        color = 'navy'
    elif os.path.exists(cont_file):
        t, v = parse_xvg(cont_file)
        ylabel = 'Contacts (<3.5Å)'
        data_type = 'Contacts'
        color = 'steelblue'
    else:
        t, v = np.array([]), np.array([])
        ylabel = 'N/A'
        data_type = 'No Data'
        color = 'gray'
    
    if len(t) > 0:
        t_ns = t / 1000
        summary[name] = {'mean': np.mean(v), 'std': np.std(v), 'max': np.max(v), 'min': np.min(v), 'type': data_type}
        
        axes[idx,0].plot(t_ns, v, color=color, lw=0.5, alpha=0.7)
        axes[idx,0].axhline(np.mean(v), c='red', ls='--', lw=2, label=f'Mean: {np.mean(v):.2f}±{np.std(v):.2f}')
        axes[idx,0].fill_between(t_ns, 0, v, alpha=0.3, color=color)
        axes[idx,0].set_xlabel('Time (ns)', fontsize=12)
        axes[idx,0].set_ylabel(ylabel, fontsize=12)
        axes[idx,0].set_title(f'{name} - {data_type}', fontweight='bold', fontsize=14)
        axes[idx,0].legend(fontsize=10)
        axes[idx,0].grid(alpha=0.3)
        axes[idx,0].set_xlim(0, max(t_ns))
        
        axes[idx,1].hist(v, bins=30, edgecolor='black', alpha=0.7, color=color)
        axes[idx,1].axvline(np.mean(v), c='red', ls='--', lw=2)
        axes[idx,1].set_xlabel(ylabel, fontsize=12)
        axes[idx,1].set_ylabel('Frequency', fontsize=12)
        axes[idx,1].set_title(f'{name} Distribution', fontsize=14)
        
        print(f'{name}: {data_type} = {np.mean(v):.2f}±{np.std(v):.2f} (Max: {np.max(v):.0f})')
    else:
        axes[idx,0].text(0.5, 0.5, 'No Data', ha='center', fontsize=20, transform=axes[idx,0].transAxes)
        axes[idx,1].text(0.5, 0.5, 'No Data', ha='center', fontsize=20, transform=axes[idx,1].transAxes)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/interaction_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'\nSaved: {WORK_DIR}/interaction_analysis.png')

## Summary

In [ ]:
print('='*75)
print('PROTEIN-LIGAND INTERACTION ANALYSIS SUMMARY')
print('='*75)
print(f'{"Complex":<25} {"Type":<12} {"Mean±Std":<18} {"Max":<8} {"Min":<8}')
print('-'*75)
for name, s in summary.items():
    mean_std = f"{s['mean']:.2f}±{s['std']:.2f}"
    print(f"{name:<25} {s['type']:<12} {mean_std:<18} {s['max']:<8.0f} {s['min']:<8.0f}")
print('='*75)

print('\nOutput files:')
for name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    folder = f'{WORK_DIR}/{name}'
    if os.path.exists(folder):
        for f in sorted(os.listdir(folder)):
            if f.endswith('.xvg'):
                print(f'  {name}/{f}')